In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv('rps_data.csv', sep=',')

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
import joblib

df = pd.read_csv('rps_data.csv')

# Обработка специальных значений
df['opp_match_wins'] = df['opp_match_wins'].replace(-1, np.nan)
df['opp_wins_known'] = (~df['opp_match_wins'].isna()).astype(int)

df['opp_match_winrate'] = df['opp_match_winrate'].apply(lambda x: np.nan if x == -0.01 else x)
df['opp_winrate_known'] = (~df['opp_match_winrate'].isna()).astype(int)

# Кодирование ходов (кроме целевого opp_move, но включая my_move для лага)
move_cols = ['prev_opp_move', 'prev2_opp_move', 'my_move']
for col in move_cols:
    df[col] = df[col].astype(str)

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df[move_cols] = encoder.fit_transform(df[move_cols])
df.rename(columns={col: col + '_enc' for col in move_cols}, inplace=True)

# Целевая переменная кодируется отдельно
target_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df['opp_move_enc'] = target_encoder.fit_transform(df[['opp_move']].astype(str))

# Исходы
outcome_map = {'win': 1, 'lose': -1, 'draw': 0, 'none': 0}
df['prev_outcome_enc'] = df['prev_outcome'].map(outcome_map)

# Лаг своего хода
df['my_prev_move_enc'] = df.groupby('match_id')['my_move_enc'].shift(1).fillna(-1)

# Признаки (без opp_move_enc, без my_move_enc)
feature_columns = [
    'round', 'score_me_before', 'score_opp_before',
    'opp_match_wins', 'opp_match_winrate', 'stake',
    'prev_outcome_enc', 'streak_draws',
    'my_prev_move_enc',
    'prev_opp_move_enc', 'prev2_opp_move_enc',
    'opp_wins_known', 'opp_winrate_known'
]

X = df[feature_columns]
y = df['opp_move_enc']

imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(X), columns=feature_columns)

# Обучение с групповой кросс-валидацией (чтобы не перемешивать раунды одного матча)
model = GradientBoostingClassifier(n_estimators=100, random_state=42)
groups = df['match_id']
cv = GroupKFold(n_splits=5)
scores = cross_val_score(model, X, y, groups=groups, cv=cv, scoring='accuracy')
print(f"CV Mean Accuracy (групповая): {scores.mean():.4f} (+/- {scores.std():.4f})")

# Обучаем на всех данных
model.fit(X, y)

# Сохраняем
joblib.dump(model, 'rps_model_fixed.pkl')
joblib.dump(encoder, 'rps_encoder_fixed.pkl')
joblib.dump(target_encoder, 'rps_target_encoder_fixed.pkl')
joblib.dump(imputer, 'rps_imputer_fixed.pkl')
joblib.dump(feature_columns, 'rps_features_fixed.pkl')
print("Исправленная модель сохранена.")

CV Mean Accuracy (групповая): 0.3149 (+/- 0.0239)
Исправленная модель сохранена.


In [3]:
import pandas as pd
import numpy as np
from collections import defaultdict, Counter

class AdaptiveRPSPredictor:
    def __init__(self):
        # Априорные вероятности первого хода (можно взять из данных)
        self.first_move_probs = {'К': 0.33, 'Н': 0.33, 'Б': 0.34}
        # Матрица переходов: prev -> текущий
        self.transitions = defaultdict(Counter)
        self.match_history = []  # ходы оппонента в текущем матче
        self.prev_opp_move = None
        
    def update(self, opp_move):
        """Обновить модель после раунда."""
        if self.prev_opp_move is not None:
            self.transitions[self.prev_opp_move][opp_move] += 1
        self.match_history.append(opp_move)
        self.prev_opp_move = opp_move
        
    def predict_proba(self):
        """Вероятности следующего хода оппонента."""
        if self.prev_opp_move is None:
            # Первый раунд — априорные
            total = sum(self.first_move_probs.values())
            return {k: v/total for k, v in self.first_move_probs.items()}
        else:
            # Марковский переход 1-го порядка
            counter = self.transitions[self.prev_opp_move]
            total = sum(counter.values())
            if total == 0:
                # Если переходов не видели — равномерно
                return {'К': 1/3, 'Н': 1/3, 'Б': 1/3}
            return {k: counter[k]/total for k in ['К', 'Н', 'Б']}
    
    def choose_move(self, strategy='max_ev'):
        probs = self.predict_proba()
        pK, pH, pB = probs['К'], probs['Н'], probs['Б']
        ev = {
            'К': pH - pB,
            'Н': pB - pK,
            'Б': pK - pH
        }
        if strategy == 'max_ev':
            return max(ev, key=ev.get)
        else:
            loss = {'К': pB, 'Н': pK, 'Б': pH}
            return min(loss, key=loss.get)

In [4]:
import pandas as pd
from collections import defaultdict, Counter

class AdaptiveRPSPredictor:
    def __init__(self):
        self.first_move_probs = {'К': 0.33, 'Н': 0.33, 'Б': 0.34}
        self.transitions = defaultdict(Counter)
        self.prev_opp_move = None
        
    def update(self, opp_move):
        if self.prev_opp_move is not None:
            self.transitions[self.prev_opp_move][opp_move] += 1
        self.prev_opp_move = opp_move
        
    def predict_proba(self):
        if self.prev_opp_move is None:
            total = sum(self.first_move_probs.values())
            return {k: v/total for k, v in self.first_move_probs.items()}
        else:
            counter = self.transitions[self.prev_opp_move]
            total = sum(counter.values())
            if total == 0:
                return {'К': 1/3, 'Н': 1/3, 'Б': 1/3}
            return {k: counter[k]/total for k in ['К', 'Н', 'Б']}
    
    def choose_move(self, strategy='max_ev'):
        probs = self.predict_proba()
        pK, pH, pB = probs['К'], probs['Н'], probs['Б']
        ev = {'К': pH - pB, 'Н': pB - pK, 'Б': pK - pH}
        if strategy == 'max_ev':
            return max(ev, key=ev.get)
        else:
            loss = {'К': pB, 'Н': pK, 'Б': pH}
            return min(loss, key=loss.get)

# Загрузка данных
df = pd.read_csv('rps_data.csv')

total = 0
correct = 0

for match_id, group in df.groupby('match_id'):
    predictor = AdaptiveRPSPredictor()
    group = group.sort_values('round')
    for _, row in group.iterrows():
        pred = predictor.choose_move()
        if pred == row['opp_move']:
            correct += 1
        total += 1
        predictor.update(row['opp_move'])

print(f"Точность адаптивной модели: {correct}/{total} = {correct/total:.4f}")

Точность адаптивной модели: 184/486 = 0.3786


In [5]:
import pandas as pd
from collections import defaultdict, Counter

class Markov2RPSPredictor:
    def __init__(self):
        self.first_move_probs = {'К': 0.33, 'Н': 0.33, 'Б': 0.34}
        # Ключ: (prev2, prev1) -> счётчик следующего хода
        self.transitions = defaultdict(Counter)
        self.history = []
        
    def update(self, opp_move):
        # Сохраняем ход после того, как он стал известен
        if len(self.history) >= 2:
            prev2, prev1 = self.history[-2], self.history[-1]
            self.transitions[(prev2, prev1)][opp_move] += 1
        elif len(self.history) == 1:
            prev1 = self.history[-1]
            self.transitions[(None, prev1)][opp_move] += 1
        # Для первого раунда переходы не сохраняем (нет предыдущего)
        self.history.append(opp_move)
        
    def predict_proba(self):
        if len(self.history) == 0:
            return self.first_move_probs.copy()
        elif len(self.history) == 1:
            prev1 = self.history[-1]
            counter = self.transitions[(None, prev1)]
            total = sum(counter.values())
            if total == 0:
                return {'К': 1/3, 'Н': 1/3, 'Б': 1/3}
            return {k: counter[k]/total for k in ['К', 'Н', 'Б']}
        else:
            prev2, prev1 = self.history[-2], self.history[-1]
            counter = self.transitions[(prev2, prev1)]
            total = sum(counter.values())
            if total == 0:
                # fallback на марков 1-го порядка
                counter = self.transitions[(None, prev1)]
                total = sum(counter.values())
                if total == 0:
                    return {'К': 1/3, 'Н': 1/3, 'Б': 1/3}
            return {k: counter[k]/total for k in ['К', 'Н', 'Б']}
    
    def choose_move(self, strategy='max_ev'):
        probs = self.predict_proba()
        pK, pH, pB = probs['К'], probs['Н'], probs['Б']
        ev = {
            'К': pH - pB,
            'Н': pB - pK,
            'Б': pK - pH
        }
        if strategy == 'max_ev':
            return max(ev, key=ev.get)
        else:
            loss = {'К': pB, 'Н': pK, 'Б': pH}
            return min(loss, key=loss.get)

# Загрузка данных
df = pd.read_csv('rps_data.csv')

total = 0
correct = 0

for match_id, group in df.groupby('match_id'):
    predictor = Markov2RPSPredictor()
    group = group.sort_values('round')
    for _, row in group.iterrows():
        pred = predictor.choose_move()
        if pred == row['opp_move']:
            correct += 1
        total += 1
        predictor.update(row['opp_move'])

print(f"Точность марковской модели 2-го порядка: {correct}/{total} = {correct/total:.4f}")

Точность марковской модели 2-го порядка: 181/486 = 0.3724
